In [ ]:
# ============================================
# 1. Introduction
# ============================================

# This notebook creates new variables (feature engineering)
# for the extended Telco Customer Churn dataset.
# It includes:
# - Tenure buckets
# - Service count
# - Engagement score
# - Billing risk score
# - CLTV normalization
# - Preparation for segmentation and modeling

# ============================================
# 2. Load libraries
# ============================================

import pandas as pd
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

sns.set(style="whitegrid")

# ============================================
# 3. Load clean dataset
# ============================================

df = pd.read_csv("../data/processed/clean_telco.csv")
df.head()

def tenure_bucket(months):
    if months < 6:
        return "0-6 months"
    elif months < 12:
        return "6-12 months"
    elif months < 24:
        return "1-2 years"
    elif months < 48:
        return "2-4 years"
    else:
        return "4+ years"

df["TenureBucket"] = df["TenureinMonths"].apply(tenure_bucket)
df["TenureBucket"].value_counts()

service_cols = [
    "PhoneService", "MultipleLines", "InternetService", "InternetType",
    "OnlineSecurity", "OnlineBackup", "DeviceProtectionPlan",
    "PremiumTechSupport", "StreamingTV", "StreamingMovies", "StreamingMusic",
    "UnlimitedData"
]

df["TotalServices"] = df[service_cols].apply(lambda row: sum(row == "Yes"), axis=1)
df["TotalServices"].describe()

engagement_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtectionPlan",
    "PremiumTechSupport"
]

df["EngagementScore"] = df[engagement_cols].apply(lambda row: sum(row == "Yes"), axis=1)
df["EngagementScore"].describe()

df["BillingRiskScore"] = (
    df["MonthlyCharge"] * 0.6 +
    df["TotalExtraDataCharges"] * 0.2 +
    df["TotalLongDistanceCharges"] * 0.2
)

df["BillingRiskScore"].describe()

scaler = MinMaxScaler()
df["CLTV_Normalized"] = scaler.fit_transform(df[["CLTV"]])
df["CLTV_Normalized"].describe()

binary_cols = [
    "PhoneService", "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtectionPlan", "PremiumTechSupport", "StreamingTV",
    "StreamingMovies", "StreamingMusic", "UnlimitedData", "PaperlessBilling"
]

for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

numeric_df = df.select_dtypes(include=["int64", "float64"])
numeric_df.head()

df.to_csv("../data/processed/features_telco.csv", index=False)
print("Dataset with features saved to data/processed/features_telco.csv")

print("""
FEATURE ENGINEERING CONCLUSIONS:

1. Tenure buckets were created for time-based analysis.
2. TotalServices was generated to measure customer complexity.
3. EngagementScore captures the use of key services.
4. BillingRiskScore models the customer's financial risk.
5. CLTV was normalized for segmentation and modeling.
6. Binary variables were converted to numeric format.
7. The dataset is ready for segmentation in 04_Segmentation.ipynb.
""")
